In [10]:
###################################################################################################

### LIBRARIES ###

import pandas as pd
import random
import ipywidgets as widgets                                  # the widget library (buttons, labels, boxes)
from IPython.display import display, clear_output             # display() shows widgets; clear_output() erases printed text
from IPython.display import Image as IPImage                  # `as` gives it a unique name, so PIL's Image can't replace it
import requests
from PIL import Image
import io
                                                              # audio libraries
import json                                                   
from google.oauth2 import service_account
from google.auth.transport.requests import Request
from IPython.display import Audio
import base64

import threading                                              # library for title screen animation

###################################################################################################

### DATAFRAME SETUP ###

animal_df = pd.read_csv('data/tv_animal_sidekicks_102926.csv')

animal_simple_df = animal_df[['name', 'show', 'species', 'image_url']]

name = animal_simple_df['name'][0]
show = animal_simple_df['show'][0]
species = animal_simple_df['species'][0]


###################################################################################################

### AUDIO SETUP ###

# Load credentials from the service account file
creds = service_account.Credentials.from_service_account_file(
    "keys/service.json",
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)

# Fetch an access token (this is what gcloud auth print-access-token returns)
creds.refresh(Request())
token = creds.token

with open('keys/keys.json', 'r') as f:
    keys = json.load(f)

def theme_song_audio():

    url = "https://aiplatform.googleapis.com/v1beta1/projects/wpp-cto-smartprod-sandbox/locations/global/interactions"

    payload = {
      "model": "lyria-3-clip-preview",
      "input": [
        {
          "type": "text",
          "text": f"Theme song for a trivia game about animal sidekicks in tv shows"
        }
      ]
    }
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    response = requests.post(url, headers=headers, json=payload)
    data = response.json()
    music_databytes = data["outputs"][2]['data']
    mp3_bytes = base64.b64decode(music_databytes)
    
    return Audio(mp3_bytes,autoplay=True)

def hint_audio(animal,show):

    url = "https://aiplatform.googleapis.com/v1beta1/projects/wpp-cto-smartprod-sandbox/locations/global/interactions"

    payload = {
      "model": "lyria-3-clip-preview",
      "input": [
        {
          "type": "text",
          "text": f"Theme song for the animal {animal} from the tv show {show}. Don't mention the species of the animal explicitly"
        }
      ]
    }
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    response = requests.post(url, headers=headers, json=payload)
    data = response.json()
    music_databytes = data["outputs"][2]['data']
    mp3_bytes = base64.b64decode(music_databytes)

    return Audio(mp3_bytes,autoplay=True)


TITLE_ANIMATION = globals().get('TITLE_ANIMATION', {'stop': None}) # keeps track of the running title animation, so re-running the cell can stop the old one

TOTAL_QUESTIONS = 12                                           # NEW: change this one number to make a round longer or shorter


###################################################################################################

### MAIN GAME CODE ###

def playGame():                                              # define the function for the game


    ####################################
        
    #### SET UP AND VARIABLE DEFINITIONS ###
    
    df = animal_simple_df                                     # simplifying variable for dataframe


    # a dictionary holding every value that changes during the game
    state = {
        'modifier': '',                                       # the exact button text that counts as the correct answer
        'success': False,                                     # True once the current question has been answered correctly
        'current': None,                                      # which game is on screen: 1 or 2
        'score': 0,                                           # number of questions answered correctly; starts at 0
        'missed': False,                                      # True if a wrong answer was clicked on the current question
        'row': None,                                          # the DataFrame row the current question was built from
        'hint_muted': False,                                  # True if the hint (not the player) turned the music off
        'question_num': 0,                                    # NEW: which question we're on (1–12); 0 before the game starts
        'used': set()                                         # NEW: index labels of animals already asked about this round
    }

    # helper function that creates one button with the standard settings
    def make_button(text, style='primary'):
        return widgets.Button(
            description=text,
            button_style=style,
            layout=widgets.Layout(width='500px')
        )

    question = widgets.Label()                                # a text widget that displays the current question; its text is set later through .value
    answer_buttons = [make_button(x) for x in 'abcd']         # list comprehension: runs make_button once per character in 'abcd', giving a list of 4 buttons
    nextButton = make_button('Next', style='')
    output = widgets.Output()
    imageBox = widgets.Output(                                # separate output area for the picture; margin adds a 20px gap on its left
                layout=widgets.Layout(margin='0 0 0 20px'))  
    scoreLabel = widgets.Label()                              # text widget that shows the current score
    progressLabel = widgets.Label()                           # NEW: shows "Question 3 of 12"
    resetButton = make_button('Start over', style='warning')  # 'warning' gives an orange button, so it stands apart from the answers
    startButton = make_button('Play', style='success')        # green so it stands out as the way in
    hintButton = make_button('🎧 Hint', style='info')          # 'info' gives a light-blue button
    hintBox = widgets.VBox([])                                # where the hint spinner, then the hint player, appear

    endText = widgets.HTML()                                  # NEW: final score + message; filled in when the round ends
    playAgainButton = make_button('Play again', style='success')   # NEW: on the end screen, leads back to the title

    # removes any hint from the screen (which also stops it playing) and re-enables the button
    def clear_hint():
        old = hintBox.children
        hintBox.children = []
        for w in old:
            w.close()                                         # free the old widgets so they don't pile up
        hintButton.disabled = False
        if state['hint_muted']:                               # the hint muted the music, so undo that
            state['hint_muted'] = False
            musicToggle.value = True                          # runs on_music_toggle: player goes back in, music starts

    # NEW: picks the end message based on the final score
    def result_message(score):
        if score <= 3:
            return "You need to brush up on your animals! Time for a new show?"
        elif score <= 8:
            return "You know your animal shows a bit, but could always learn more!"
        else:
            return "Wow, you know a surprising amount about animals in TV shows! Time to touch grass maybe?"

    ####################################
        
    #### AUDIO SOUND EFFECTS AND GAME MUSIC ###
    
    with open('Data/Audio/ding.mp3', 'rb') as f:              # 'rb' = read the raw bytes of the file
        ding_bytes = f.read()
    with open('Data/Audio/buzzer.mp3', 'rb') as f:
        buzzer_bytes = f.read()

    sfxBox = widgets.Box()                                    # empty, invisible slot that sound effects get dropped into

    # plays a sound by swapping a brand-new audio widget into sfxBox
    def play_sfx(data):
        for old in sfxBox.children:
            old.close()                                       # clean up the previous sound's widget so they don't pile up
        sfxBox.children = [widgets.Audio(
            value=data,
            format='mp3',                                     # use 'wav' if your files are .wav
            autoplay=True,                                    # a new widget autoplays as soon as it appears
            loop=False,
            controls=False                                    # no visible player
        )]

    gameMusic = widgets.Audio.from_file(
        'Data/audio/jeopardy_theme.mp3',                      # change to your file's path
        autoplay=True,                                        # start as soon as the game screen appears
        loop=True,                                            # start over when the track ends, so it never goes silent
        controls=False                                        # hidden player; set True if you want a pause/volume control
    )

    musicBox = widgets.Box([gameMusic])                       # the music's own slot; empty = muted

    musicToggle = widgets.ToggleButton(
        value=True,                                           # True = music on
        description='🎵 Trivia music: on',
        layout=widgets.Layout(width='180px')                  # small, so it doesn't compete with the answer buttons
    )

    # runs whenever the toggle is clicked; `change['new']` is the toggle's new value
    def on_music_toggle(change):
        if change['new']:
            musicBox.children = [gameMusic]                   # put the player back, so autoplay starts it again
            musicToggle.description = '🎵 Trivia music: on'
        else:
            musicBox.children = []                            # remove the player, so the browser stops the music
            musicToggle.description = '🔇 Trivia music: off'

    musicToggle.observe(on_music_toggle, names='value')

    ####################################
        
    #### TITLE SCREEN ###

    animal_emojis = list("🐶🐱🐰🦜🐢🐴🐷🐭🦊🐸🐮🐵🐧🦁🐯🐨🐼🐹🦉🐍")   # every animal that can appear in the rows

    banner = """█████  ████   ███  █   █  ███   ███ 
  █    █   █   █   █   █   █   █   █
  █    ████    █   █   █   █   █████
  █    █  █    █    █ █    █   █   █
  █    █   █  ███    █    ███  █   █"""

    # builds the full title HTML from two lists of 10 emojis
    def build_title(top, bottom):
        return f"""
<pre style="font-family: monospace; line-height: 1.1; font-size: 14px;">
   {' '.join(top)}

{banner}

   {' '.join(bottom)}
</pre>
<h3>Animal Characters In Popular Media</h3>
"""

    title = widgets.HTML(build_title(animal_emojis[:10], animal_emojis[10:]))   # starts with your original layout

    # runs in the background: every 0.1s, swap one random emoji in each row
    def animate_title(stop):
        top = animal_emojis[:10]
        bottom = animal_emojis[10:]
        while not stop.is_set():                              # keep going until someone flips the stop switch
            for row in (top, bottom):
                row[random.randrange(10)] = random.choice(animal_emojis)   # replace one random spot with a random animal
            title.value = build_title(top, bottom)            # updating .value re-renders the widget in the browser
            stop.wait(0.1)                                    # like time.sleep(0.1), but wakes up immediately if stop is set

    def stop_title_animation():
        if TITLE_ANIMATION['stop']:
            TITLE_ANIMATION['stop'].set()                     # tells the running loop to end

    def start_title_animation():
        stop_title_animation()                                # never run two animations at once
        TITLE_ANIMATION['stop'] = threading.Event()           # a fresh on/off switch for this run
        threading.Thread(target=animate_title,
                         args=(TITLE_ANIMATION['stop'],),
                         daemon=True).start()                 # daemon=True: the thread dies with the kernel

    #############################
    
    ### LOADING ICON ###
    
    loader_template = """
<style>
  .theme-loader {
    width: 22px; height: 22px;
    border: 3px solid #ddd;
    border-top-color: #555;
    border-radius: 50%;
    animation: theme-spin 0.9s linear infinite;
  }
  @keyframes theme-spin { to { transform: rotate(360deg); } }
</style>
<div style="display:flex; align-items:center; gap:10px; padding:8px 0;">
  <div class="theme-loader"></div>
  <span>MESSAGE</span>
</div>
"""

    # builds a spinner with any message; .replace() is used instead of an f-string because the CSS has { } in it
    def make_loader(message):
        return widgets.HTML(loader_template.replace('MESSAGE', message))

    loading = make_loader('Loading the game, please hold...')   # the title-screen spinner
    
    songPlayer = widgets.Output()  

    #############################
    
    ### BUTTONS AND FUNCTIONS ###
    
    # updates the score label to match state['score']
    def updateScore():
        scoreLabel.value = f"Score: {state['score']}"

    # helper that turns one animal's row into button text like "Kit, from Charmed"
    def label(row):
        return f"{row['name']}, from {row['show']}"

    # NEW: picks a random animal that hasn't been asked about yet this round
    def pickRow():
        unused = df.drop(index=list(state['used']))           # drop() removes rows by index label: every animal already used
        if unused.empty:                                      # safety net: if the dataset runs out, allow repeats again
            state['used'].clear()
            unused = df
        row = unused.sample(1).iloc[0]
        state['used'].add(row.name)                           # row.name is the row's index label in the DataFrame
        return row

    # builds the 4 options for game1: 3 wrong values from a column plus the correct one
    def pickOptions(column, correct):

        # step by step, inside to outside:
        #   df[column] != correct   -> a "boolean mask": a Series of True/False, one per row, True wherever the value is NOT the correct answer
        #   df.loc[mask, column]    -> .loc selects by labels/conditions: rows where the mask is True, and only the given column, giving a Series of wrong answers
        #   .unique()               -> removes duplicates (so "Dog" appears once, not five times); returns a NumPy array, not a Series
        #   .tolist()               -> converts that array into a plain Python list

        wrong = df.loc[df[column] != correct, column].unique().tolist()
        options = random.sample(wrong, 3) + [correct]         # random.sample(list, 3) picks 3 *different* items at random; then add the correct answer
        random.shuffle(options)                               # shuffle the list in place so the correct answer isn't always the last button
        return options

    # builds the 4 options for game2: 3 animals of other species (one per species) plus the correct animal
    def pickAnimalOptions(row):
        others = (                                            # wrapping the expression in parentheses lets us split a "method chain" across several lines
            df[df['species'] != row['species']]               # boolean mask again: keep only rows whose species differs from the correct animal's (df[mask] filters rows and keeps every column)
                  .sample(frac=1)                             # .sample(frac=1) takes a random sample of 100% of the rows, which in practice shuffles the whole DataFrame
                  .drop_duplicates('species')                 # drop_duplicates('species') keeps only the FIRST row for each species value; because we shuffled first, which dog/cat/etc. is "first" is random each time
                  .head(3)                                    # .head(3) keeps the first 3 rows of what's left: 3 animals, 3 different species
        )

        # .iterrows() loops over a DataFrame one row at a time, yielding (index, row) pairs;
        # `_` is a Python convention for "a value I don't need" (here, the index);
        # the list comprehension labels each row, then we add the correct animal's label

        options = [label(r) for _, r in others.iterrows()] + [label(row)]
        random.shuffle(options)                               # shuffle so the correct animal lands on a random button
        return options                                        # hand back the 4 shuffled labels

    # loads a question onto the screen: question text, correct answer, and 4 options
    def setQuestion(text, correct, options):
        for button, option in zip(answer_buttons, options):   # zip() pairs items from two lists side by side: (button1, option1), (button2, option2), ...
            button.description = option                       # put this option's text on the button
            button.button_style = 'primary'                   # reset the color to blue (it may be green/red from the last question)

        # everything indented under this line is directed into the output box
        with output:
            clear_output()                                    # erase the previous "That's right!" / "Incorrect" message
        with imageBox:
            clear_output()                                    # erase the previous picture
        state['success'] = False                              # this new question hasn't been answered yet
        state['missed'] = False                               # ...and hasn't been missed yet either
        state['modifier'] = correct                           # remember which button text is correct
        question.value = text                                 # display the question text in the Label widget
        clear_hint()                                          # new question: remove the old hint, re-enable the button

        state['question_num'] += 1                            # NEW: every new question on screen counts as the next level
        progressLabel.value = f"Question {state['question_num']} of {TOTAL_QUESTIONS}"
        # NEW: on the last question, the Next button leads to the results instead
        nextButton.description = 'See my score' if state['question_num'] == TOTAL_QUESTIONS else 'Next'
        

###################################################################################################

### GAME 1 MECHANICS ###

    # game 1: show an animal, ask for its species
    def game1():
        state['current'] = 1                                  # record that game 1 is on screen, so Next knows to switch to game 2
        row = pickRow()                                       # NEW: a random animal not yet used this round
        state['row'] = row                                    # remember it, so the answer handler can show its image
        # load the question: text, correct answer (that row's species), and the 4 options
        setQuestion(f"What species is the animal {row['name']} from the show {row['show']}?",
                    row['species'], pickOptions('species', row['species']))

###################################################################################################

### GAME 2 MECHANICS ###

    # game 2: name a species, ask which animal belongs to it
    def game2():
        state['current'] = 2                                  # record that game 2 is on screen
        row = pickRow()                                       # NEW: a random animal not yet used this round; its species becomes the question
        state['row'] = row   
        
        setQuestion(f"Which of these is a {row['species'].lower()}?",
                    label(row), pickAnimalOptions(row))       # correct answer: this animal's label, e.g. "Kit, from Charmed"

###################################################################################################

### BUTTON MECHANICS ###

    # NEW: swaps the game out for the results screen
    def show_end_screen():
        clear_hint()                                          # stop any hint and restore the music setting
        score = state['score']
        endText.value = f"""
<h2>You scored {score} out of {TOTAL_QUESTIONS}</h2>
<p style="font-size: 16px;">{result_message(score)}</p>
"""
        screen.children = [endScreen]                         # removing gameUI also stops the trivia music

    # runs whenever an answer button is clicked; ipywidgets passes the clicked button in as `b`
    def on_answer_click(b):
        if state['success']:                                  # if this question was already answered correctly, ignore further clicks
            return
        with output:
            clear_output()
            if b.description == state['modifier']:            # compare the clicked button's text against the stored correct answer
                b.button_style = 'success'                    # correct: turn the button green
                play_sfx(ding_bytes)                          # correct: ding!
                state['success'] = True                       # remember that this question is solved (this unlocks Next)
                if not state['missed']:                       # only award a point if there were no wrong guesses first
                    state['score'] += 1
                    updateScore()
                    print("That's right! Press Next to continue.")
                else:
                    print("That's right, but no point this time. Press Next to continue.")

                url = state['row']['image_url']               # look up the image URL for this question's animal
                if pd.notna(url):
                    with imageBox:                            # direct the image into the right-hand box instead of the message box
                        display(IPImage(url=url, width=300))
            else:
                b.button_style = 'danger'
                play_sfx(buzzer_bytes)                        # wrong: buzzer
                state['missed'] = True
                print("Incorrect, try again")

    # runs whenever the Next button is clicked
    def on_next_click(b):
        if not state['success']:                              # `not` flips True/False: this checks "has the question NOT been solved yet?"
            with output:                                      # send the warning into the output box
                clear_output()                                # remove the previous message
                print("Answer the question first!")
            return                                            # stop without switching games

        if state['question_num'] >= TOTAL_QUESTIONS:          # NEW: that was the last question
            show_end_screen()
            return                                            # stop here instead of loading another question

        game2() if state['current'] == 1 else game1()         # conditional expression (a one-line if/else): run game2 if we're on game 1, otherwise game1

    # runs whenever the "start over" or "play again" button is clicked: resets the score and returns to the title screen
    def on_reset_click(b):
        state['score'] = 0                                    # set the score back to 0
        updateScore()                                         # refresh the label so it reads "Score: 0" next time the game shows
        with output:
            clear_output()                                    # erase the previous message
        with imageBox:
            clear_output()                                    # erase the previous picture
        screen.children = [titleScreen]                       # swap the game UI out and the title screen back in
        start_title_animation()                               # title screen is back, so animate it again
        clear_hint()                                          # remove any hint and restore the music setting

    # runs when the start button is clicked: swaps the title screen for the game
    def on_start_click(b):
        stop_title_animation()                                # title screen is leaving, so stop animating it
        state['question_num'] = 0                             # NEW: fresh round; setQuestion bumps this to 1
        state['score'] = 0                                    # NEW: make sure every round starts from zero
        state['used'].clear()                                 # NEW: every animal is available again
        game1()                                               # load the first question
        updateScore()                                         # show "Score: 0"
        screen.children = [gameUI]                            # replace the title screen with the game UI

    # runs when the hint button is clicked: generates a song about the current animal
    def on_hint_click(b):
        row = state['row']
        clear_hint()
        b.disabled = True
        nextButton.disabled = True                            # lock Next and Start over while the hint composes
        resetButton.disabled = True
        hintBox.children = [make_loader('Generating your audio hint, please hold...!')]

        try:
            audio = hint_audio(row['name'], row['show'])
            if musicToggle.value:
                state['hint_muted'] = True
                musicToggle.value = False
            hintPlayer = widgets.Output()
            with hintPlayer:
                display(audio)
            hintBox.children = [hintPlayer]                   # hint is now on screen and playing
        except Exception as e:
            hintBox.children = [widgets.HTML(f"<b>Couldn't load the hint:</b> {e}")]
            b.disabled = False
        finally:
            nextButton.disabled = False                       # always unlock, whether the hint worked or failed
            resetButton.disabled = False

    for button in answer_buttons:                             # loop over the 4 answer buttons
        button.on_click(on_answer_click)
    nextButton.on_click(on_next_click)                        # NOT indented under the loop: register it once
    resetButton.on_click(on_reset_click)
    startButton.on_click(on_start_click)
    hintButton.on_click(on_hint_click)
    playAgainButton.on_click(on_reset_click)                  # NEW: "Play again" does the same as "Start over"

###################################################################################################

### GAME INITIALIZATION ###

    titleScreen = widgets.VBox([title, songPlayer, startButton])   # the title screen: heading + theme song + start button

    leftColumn = widgets.VBox([musicBox, sfxBox, musicToggle, progressLabel, scoreLabel,   # NEW: progressLabel
                               question, *answer_buttons, hintButton, hintBox,
                               nextButton, resetButton, output])   # everything in the game screen, stacked vertically

    gameUI = widgets.HBox([leftColumn, imageBox])             # HBox places its children side by side: controls on the left, picture on the right

    endScreen = widgets.VBox([title, endText, playAgainButton])   # NEW: results screen: banner + score + message + button
    
    screen = widgets.VBox([loading])                          # start with the spinner in the container
    display(screen)                                           # show it right away, before the slow API call

    try:
        song = theme_song_audio()                             # the slow part: generate + download the song
        with songPlayer:
            display(song)                                     # put the player into its slot on the title screen
    except Exception as e:
        with songPlayer:
            print(f"Couldn't load the theme song: {e}")       # still let people play if the song fails

    screen.children = [titleScreen] 
    start_title_animation()                                   # song is loaded and title is showing: start the animation

###################################################################################################

### GAME START! ###

playGame()                                                    # start the game upon running this code block